In [0]:
import sys
sys.path.append("../src")
from pyspark.sql import functions as F
import observability.metrics 
import observability.runner
import importlib
importlib.reload(observability.metrics)
importlib.reload(observability.runner)
from observability.metrics import RowCountMetric, NullRateMetric, DistinctCountMetric
from observability.runner import ObservabilityRunner

In [0]:

metrics_config = [
    {
        "metric": "row_count",
        "enabled": True
    },
    {
        "metric": "null_rate",
        "enabled": True,
        "column_name": "amount"
    },
    {
        "metric": "distinct_count",
        "enabled": False,
        "column_name": "country"
    }
]

metric_classes = {
    "row_count": RowCountMetric,
    "null_rate": NullRateMetric,
    "distinct_count": DistinctCountMetric
}

metrics = []

for item in metrics_config:

    if not item["enabled"]:
        continue
    elif item["enabled"] and "column_name" not in item:
        metric_name = metric_classes[item["metric"]]()
        metrics.append(metric_name)
    elif item["enabled"] and "column_name" in item:
        metric_name = metric_classes[item["metric"]](item["column_name"])
        metrics.append(metric_name)
        

In [0]:
orders_df = (
            spark.range(1_000_000)
            .withColumnRenamed("id", "order_id")
            .withColumn(
                "country",
                F.when(F.col("order_id") % 10 == 0, None)
                .when(F.col("order_id") % 3 == 0, "PL")
                .when(F.col("order_id") % 3 == 1, "DE")
                .otherwise("UK")
                )
            .withColumn(
                "amount",
                F.when(F.col("order_id") % 20 == 0, None)
                .otherwise((F.rand() * 1000).cast("double"))
            )
            )

display(orders_df)
orders_df.printSchema()
orders_df.select(F.spark_partition_id().alias("partition_id")).groupBy("partition_id").count().orderBy("partition_id").show()

In [0]:
# row_count = RowCountMetric()
# print(f"Row number: {row_count.calculate(orders_df)}")

# null_rate = NullRateMetric("amount")
# print(f"Null rate: {null_rate.calculate(orders_df)}")

# distinct_count = DistinctCountMetric("country")
# print(f"Distinct row count: {distinct_count.calculate(orders_df)}")

observability_runner = ObservabilityRunner(metrics, orders_df)
observability_runner.run()
